# 02 — Prepare the YOLO datasets

Turns the COCO annotations into two YOLO datasets with a **real** validation
split (the old pipeline validated on its own training images — that is the
bug this rebuild fixes).

Protocol:
- 85 / 15 train / val split of the training images, stratified so rare disease
  classes appear on both sides; fixed seed 42.
- The official 50-image validation set is **never** included; it stays locked
  as the final judge (`dentex_week3/evaluate.py`).
- Images are hardlinked (no duplicated disk), labels written as
  `class cx cy w h`.
- The exact split lists are written to `splits.json`; the same seed and code
  reproduce them byte-for-byte on Kaggle.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
import dentex_data as dd

DATA_ROOT = Path("/home/sayfeldin/Projects/dental-model-research")
OUT_ROOT = REPO_ROOT / "datasets/dentex_yolo"
SEED, VAL_FRAC = 42, 0.15

qe = dd.load_coco(DATA_ROOT / "data/raw/dentex/quadrant_enumeration/train_quadrant_enumeration.json")
qed = dd.load_coco(DATA_ROOT / "data/raw/dentex/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json")
print("loaded QE:", len(qe["images"]), "| QED:", len(qed["images"]))

In [ ]:
# ---- Tooth detector: 32 FDI classes, trained on the 634 QE panoramas ----
tooth_labels = dd.yolo_labels(qe, dd.tooth_class)

def tooth_signature(name):
    # balance by how many teeth are annotated (sparse vs full dentition)
    n = len(tooth_labels.get(name, []))
    return "sparse" if n < 20 else ("dense" if n > 30 else "normal")

tooth_names = dd.image_names(qe)
tooth_train, tooth_val = dd.stratified_split(tooth_names, tooth_signature, VAL_FRAC, SEED)
print(f"tooth: {len(tooth_train)} train / {len(tooth_val)} val")

In [ ]:
# ---- Disease detector: 4 classes, trained on all 705 QED panoramas ----
disease_labels = dd.yolo_labels(qed, dd.disease_class)

_by_image = {}
for ann in qed["annotations"]:
    _by_image.setdefault(ann["image_id"], set()).add(ann["category_id_3"])
_id_to_name = {i["id"]: i["file_name"] for i in qed["images"]}

def disease_signature(name):
    ids = _by_image.get(next(k for k, v in _id_to_name.items() if v == name), set())
    return tuple(sorted(ids))

disease_names = dd.image_names(qed)
disease_train, disease_val = dd.stratified_split(disease_names, disease_signature, VAL_FRAC, SEED)
print(f"disease: {len(disease_train)} train / {len(disease_val)} val")

In [ ]:
qe_dir = DATA_ROOT / "data/raw/dentex/quadrant_enumeration/xrays"
qed_dir = DATA_ROOT / "data/raw/dentex/training_data/quadrant-enumeration-disease/xrays"

tooth_stats = dd.build_yolo_dataset(
    OUT_ROOT, "tooth", qe_dir, tooth_labels, tooth_train, tooth_val,
    dd.tooth_class_names())
disease_stats = dd.build_yolo_dataset(
    OUT_ROOT, "disease", qed_dir, disease_labels, disease_train, disease_val,
    dd.DISEASE_NAMES)

for name, stats in (("tooth", tooth_stats), ("disease", disease_stats)):
    print(f"{name:8s} train {stats['train_images']:4d} imgs / {stats['train_boxes']:6d} boxes | "
          f"val {stats['val_images']:3d} imgs / {stats['val_boxes']:5d} boxes | "
          f"missing {len(stats['missing_images'])}")

In [ ]:
dd.write_splits(OUT_ROOT / "splits.json", {
    "seed": SEED, "val_fraction": VAL_FRAC,
    "tooth": {"train": tooth_train, "val": tooth_val},
    "disease": {"train": disease_train, "val": disease_val},
})
print("wrote", OUT_ROOT / "splits.json")

In [ ]:
# ---- Integrity checks ----
val_files = {i["file_name"] for i in dd.load_coco(DATA_ROOT / "data/downloads/validation_triple.json")["images"]}

assert not (set(tooth_train + tooth_val) & val_files), "tooth split leaks val images"
assert not (set(disease_train + disease_val) & val_files), "disease split leaks val images"
assert not (set(tooth_train) & set(tooth_val)), "tooth train/val overlap"
assert not (set(disease_train) & set(disease_val)), "disease train/val overlap"

for task, names, n_classes in (("tooth", tooth_train + tooth_val, 32),
                               ("disease", disease_train + disease_val, 4)):
    bad = 0
    for fname in names:
        for split in ("train", "val"):
            p = OUT_ROOT / task / "labels" / split / (Path(fname).stem + ".txt")
            if not p.exists():
                continue
            for line in p.read_text().splitlines():
                parts = line.split()
                assert len(parts) == 5, f"bad line in {p}: {line}"
                cls = int(parts[0])
                assert 0 <= cls < n_classes, f"class {cls} out of range in {p}"
                assert all(0.0 <= float(v) <= 1.0 for v in parts[1:]), f"unnormalised box in {p}"
                bad += 1
    print(f"{task:8s} checked {bad:6d} label rows - OK")

print("\nAll integrity checks passed.")

In [ ]:
# ---- Final summary ----
for task in ("tooth", "disease"):
    root = OUT_ROOT / task
    print(f"--- {task} ---")
    print((root / "data.yaml").read_text())
    for split in ("train", "val"):
        imgs = list((root / "images" / split).iterdir())
        labels = list((root / "labels" / split).iterdir())
        print(f"  {split}: {len(imgs)} images, {len(labels)} label files")
    print()

## Output

```
datasets/dentex_yolo/            (gitignored)
├── splits.json                  exact file lists + seed
├── tooth/
│   ├── data.yaml                32 FDI classes
│   ├── images/{train,val}/
│   └── labels/{train,val}/
└── disease/
    ├── data.yaml                4 disease classes
    ├── images/{train,val}/
    └── labels/{train,val}/
```

**Next:** `03_train_tooth_detector.ipynb` on Kaggle (mounts the public DENTEX
dataset and rebuilds the identical split with the same seed).